In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
CUDA环境检查和诊断Notebook
用于检查PyTorch GPU兼容性问题
"""

# %% [markdown]
# # CUDA环境检查和诊断
# 
# 这个notebook将帮助你诊断为什么Phase 2使用的是CPU而不是GPU

# %% [markdown]
# ## 1. 基础环境信息

# %%
import sys
import platform
import subprocess

print("=== 系统信息 ===")
print(f"Python版本: {sys.version}")
print(f"系统平台: {platform.platform()}")
print(f"系统架构: {platform.machine()}")

# %% [markdown]
# ## 2. PyTorch和CUDA检查

# %%
try:
    import torch
    print("=== PyTorch信息 ===")
    print(f"PyTorch版本: {torch.__version__}")
    print(f"CUDA是否可用: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA版本: {torch.version.cuda}")
        print(f"cuDNN版本: {torch.backends.cudnn.version()}")
        print(f"GPU数量: {torch.cuda.device_count()}")
        
        for i in range(torch.cuda.device_count()):
            print(f"\nGPU {i}:")
            print(f"  名称: {torch.cuda.get_device_name(i)}")
            props = torch.cuda.get_device_properties(i)
            print(f"  计算能力: {props.major}.{props.minor}")
            print(f"  总内存: {props.total_memory / 1e9:.2f} GB")
            print(f"  多处理器数量: {props.multi_processor_count}")
    else:
        print("\n⚠️ CUDA不可用！")
        print("可能的原因：")
        print("1. 没有安装GPU版本的PyTorch")
        print("2. NVIDIA驱动没有正确安装")
        print("3. CUDA版本不兼容")
        
except ImportError:
    print("❌ PyTorch未安装")

# %% [markdown]
# ## 3. NVIDIA驱动检查

# %%
print("=== NVIDIA驱动信息 ===")
try:
    # 检查nvidia-smi
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ nvidia-smi可用")
        # 提取驱动版本
        for line in result.stdout.split('\n'):
            if 'Driver Version' in line:
                print(line.strip())
                break
    else:
        print("❌ nvidia-smi不可用")
except FileNotFoundError:
    print("❌ nvidia-smi未找到，可能NVIDIA驱动未安装")

# 简化的nvidia-smi输出
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', 
                           '--format=csv,noheader'], capture_output=True, text=True)
    if result.returncode == 0:
        print("\nGPU详细信息:")
        print(result.stdout)
except:
    pass

# %% [markdown]
# ## 4. 检查PyTorch构建信息

# %%
if 'torch' in locals():
    print("=== PyTorch构建配置 ===")
    print(f"是否使用CUDA构建: {torch.cuda.is_available()}")
    print(f"PyTorch CUDA编译版本: {torch._C._cuda_getCompiledVersion() if hasattr(torch._C, '_cuda_getCompiledVersion') else 'N/A'}")
    
    # 检查当前PyTorch是否是CPU版本
    if '+cu' in torch.__version__:
        print(f"✅ 检测到CUDA版本的PyTorch: {torch.__version__}")
    elif '+cpu' in torch.__version__:
        print(f"⚠️ 检测到CPU版本的PyTorch: {torch.__version__}")
    else:
        print(f"⚠️ 无法确定PyTorch版本类型: {torch.__version__}")

# %% [markdown]
# ## 5. 测试GPU计算

# %%
if 'torch' in locals() and torch.cuda.is_available():
    print("=== GPU计算测试 ===")
    
    # 创建测试张量
    device = torch.device('cuda')
    print(f"使用设备: {device}")
    
    try:
        # 简单计算测试
        x = torch.randn(1000, 1000).to(device)
        y = torch.randn(1000, 1000).to(device)
        
        # 执行矩阵乘法
        import time
        start = time.time()
        z = torch.matmul(x, y)
        torch.cuda.synchronize()  # 等待GPU完成
        gpu_time = time.time() - start
        
        print(f"✅ GPU计算成功!")
        print(f"矩阵乘法 (1000x1000) 耗时: {gpu_time:.4f} 秒")
        
        # 对比CPU时间
        x_cpu = x.cpu()
        y_cpu = y.cpu()
        start = time.time()
        z_cpu = torch.matmul(x_cpu, y_cpu)
        cpu_time = time.time() - start
        
        print(f"CPU耗时: {cpu_time:.4f} 秒")
        print(f"GPU加速比: {cpu_time/gpu_time:.2f}x")
        
    except Exception as e:
        print(f"❌ GPU计算失败: {e}")

# %% [markdown]
# ## 6. 检查其他深度学习库

# %%
# 检查其他可能冲突的库
libraries = ['tensorflow', 'jax', 'mxnet']

print("=== 其他深度学习库 ===")
for lib in libraries:
    try:
        module = __import__(lib)
        print(f"✓ {lib} 已安装: {module.__version__}")
    except ImportError:
        print(f"✗ {lib} 未安装")

# %% [markdown]
# ## 7. 环境变量检查

# %%
import os

print("=== CUDA相关环境变量 ===")
cuda_vars = ['CUDA_HOME', 'CUDA_PATH', 'CUDA_VISIBLE_DEVICES', 'LD_LIBRARY_PATH']

for var in cuda_vars:
    value = os.environ.get(var, '未设置')
    print(f"{var}: {value}")

# 检查是否有限制GPU可见性
if 'CUDA_VISIBLE_DEVICES' in os.environ:
    print("\n⚠️ 注意: CUDA_VISIBLE_DEVICES 已设置，可能限制了GPU可见性")

# %% [markdown]
# ## 8. 修复建议

# %%
print("=== 修复建议 ===")

if 'torch' not in locals():
    print("1. 首先安装PyTorch")
elif not torch.cuda.is_available():
    print("根据检查结果，建议：")
    print("\n1. 重新安装GPU版本的PyTorch:")
    print("   首先卸载当前版本:")
    print("   pip uninstall torch torchvision torchaudio")
    print("\n   然后根据你的CUDA版本安装:")
    print("   - CUDA 11.8: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
    print("   - CUDA 12.1: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
    print("\n2. 检查NVIDIA驱动是否正确安装")
    print("   运行: nvidia-smi")
    print("\n3. 确保CUDA toolkit已安装")
    print("   查看: nvcc --version")

# %% [markdown]
# ## 9. 测试Phase 2代码的设备选择

# %%
# 模拟Phase 2的设备选择逻辑
if 'torch' in locals():
    print("=== Phase 2 设备选择模拟 ===")
    
    # 这是Phase 2中的逻辑
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Phase 2 将使用设备: {device}")
    
    # 创建一个简单的网络测试
    if torch.cuda.is_available():
        # 模拟DeepNetworkUtils
        class TestModel(torch.nn.Module):
            def __init__(self):
                super().__init__()
                self.fc = torch.nn.Linear(341, 4096)
            
            def forward(self, x):
                return self.fc(x)
        
        model = TestModel().to(device)
        test_input = torch.randn(32, 341).to(device)
        
        try:
            output = model(test_input)
            print(f"✅ 模型成功在 {device} 上运行")
            print(f"输入形状: {test_input.shape}, 输出形状: {output.shape}")
        except Exception as e:
            print(f"❌ 模型运行失败: {e}")

# %% [markdown]
# ## 10. 快速修复脚本

# %%
print("=== 快速修复命令 ===")
print("\n如果确认是PyTorch版本问题，运行以下命令：")
print("```bash")
print("# 1. 卸载当前PyTorch")
print("pip uninstall torch torchvision torchaudio -y")
print("\n# 2. 安装GPU版本 (根据nvidia-smi显示的CUDA版本选择)")
print("# CUDA 11.8:")
print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
print("\n# CUDA 12.1:")
print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
print("\n# 3. 验证安装")
print("python -c \"import torch; print(torch.cuda.is_available())\"")
print("```")

